### **Import the Google Maps data folder**

In [11]:
import os
import glob

# =========================================================
# (0) PATH + SETTINGS  (ALL REGIONS) + QUICK STRUCTURE CHECK
# =========================================================
ALL_REGIONS_ROOT = r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Google Maps"

TEXT_COL = "Text_TR"  # العمود المراد العمل عليه
STAR_CANDIDATES = {"stars", "Stars"}

print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir:", os.path.isdir(ALL_REGIONS_ROOT))

# جلب مجلدات المناطق
region_dirs = sorted([
    d for d in glob.glob(os.path.join(ALL_REGIONS_ROOT, "*"))
    if os.path.isdir(d)
])

print("Region folders found:", len(region_dirs))
print("Regions:", [os.path.basename(d) for d in region_dirs])

# =========================================================
# عرض سريع لأول 3 مناطق (كم مدينة داخل كل منطقة)
# =========================================================
for rd in region_dirs[:3]:
    city_dirs = sorted([
        d for d in glob.glob(os.path.join(rd, "*"))
        if os.path.isdir(d)
    ])
    print(f"  - {os.path.basename(rd)}: cities={len(city_dirs)} (sample: {[os.path.basename(x) for x in city_dirs[:5]]})")


[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir: True
Region folders found: 5
Regions: ['( المنطقة الجنوبية ) Google Maps Data -  After Cleaning', '( المنطقة الشرقية ) Google Maps Data -  After Cleaning', '( المنطقة الشمالية ) Google Maps Data -  After Cleaning', '( المنطقة الغربية ) Google Maps Data -  After Cleaning', '( المنطقة الوسطى ) Google Maps Data -  After Cleaning']
  - ( المنطقة الجنوبية ) Google Maps Data -  After Cleaning: cities=4 (sample: ['منطقة الباحة', 'منطقة جازان', 'منطقة عسير', 'منطقة نجران'])
  - ( المنطقة الشرقية ) Google Maps Data -  After Cleaning: cities=6 (sample: ['الاحساء', 'الجبيل', 'الخبر', 'الدمام', 'الظهران'])
  - ( المنطقة الشمالية ) Google Maps Data -  After Cleaning: cities=5 (sample: ['تبوك', 'محافظة العلا', 'منطقة الجوف', 'منطقة الحدود الشمالية - عرعر', 'منطقة حائل'])


In [13]:
import os
import re
import glob
import pandas as pd

# =========================================================
# Helpers: extract region name (inside parentheses) + city
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def extract_region_from_folder(region_folder_name: str) -> str:
    """
    يستخرج النص بين ( ) من اسم مجلد المنطقة.
    مثال: "( المنطقة الغربية ) Google Maps Data - After Cleaning" -> "المنطقة الغربية"
    إذا لم توجد أقواس يرجع اسم المجلد نفسه.
    """
    m = PAREN_RE.search(region_folder_name)
    if m:
        return m.group(1).strip()
    return region_folder_name.strip()

def list_region_folders(root_folder: str):
    region_dirs = sorted([
        d for d in glob.glob(os.path.join(root_folder, "*"))
        if os.path.isdir(d)
    ])
    return region_dirs

def list_city_folders(region_dir: str):
    return sorted([
        d for d in glob.glob(os.path.join(region_dir, "*"))
        if os.path.isdir(d)
    ])

def list_files_in_city(city_dir: str):
    return (
        glob.glob(os.path.join(city_dir, "*.xlsx")) +
        glob.glob(os.path.join(city_dir, "*.xls")) +
        glob.glob(os.path.join(city_dir, "*.csv"))
    )

def read_any_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)

    elif ext == ".csv":
        # ترميزات عربية شائعة
        try:
            df = pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="cp1256")

    else:
        raise ValueError(f"Unsupported extension: {ext}")

    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_star_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    - يقبل Stars / stars / ... ويوحّدها إلى عمود اسمه 'Stars' إن وجد.
    - لا يغير القيم الآن.
    """
    colmap = {c.lower(): c for c in df.columns}

    # إذا موجود Stars بالاسم الصحيح خلاص
    if "Stars" in df.columns:
        return df

    # إذا موجود stars بحروف صغيرة
    if "stars" in colmap:
        df = df.rename(columns={colmap["stars"]: "Stars"})
        return df

    # مرونة إضافية
    for cand in ["Stars", "stars"]:
        key = cand.lower()
        if key in colmap:
            df = df.rename(columns={colmap[key]: "Stars"})
            return df

    return df  # لم نجد عمود نجوم

In [4]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG
# =========================
ROOT_DIR = Path(r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Google Maps")  # عدله لمسارك
FILE_GLOB = "**/*.xlsx"
SOURCE_NAME = "GoogleMaps"

def parse_path_metadata(file_path: Path):
    """
    يتوقع مسار مثل:
    ROOT / ( المنطقة الجنوبية ) ... / منطقة الباحة / اسم_المكان_textready.xlsx
    """
    parts = file_path.relative_to(ROOT_DIR).parts

    # parts[0] غالبًا مجلد المنطقة الكبرى: "( المنطقة الجنوبية ) ..."
    macro_region_folder = parts[0] if len(parts) > 0 else "UNKNOWN_MACRO"

    # parts[1] غالبًا المدينة/المحافظة/التقسيم داخل المنطقة
    subfolder = parts[1] if len(parts) > 1 else "UNKNOWN_SUB"

    place_file = file_path.stem  # اسم الملف بدون الامتداد
    place_name = place_file.replace("_textready", "").strip()

    return macro_region_folder, subfolder, place_name

def load_all_google_maps(root_dir: Path):
    all_files = sorted(root_dir.glob(FILE_GLOB))
    frames = []

    for fp in all_files:
        try:
            df = pd.read_excel(fp)

            macro_region, subfolder, place_name = parse_path_metadata(fp)

            df["source"] = SOURCE_NAME
            df["macro_region"] = macro_region
            df["subfolder"] = subfolder
            df["place_name"] = place_name
            df["file_path"] = str(fp)

            frames.append(df)

        except Exception as e:
            print(f"⚠️ Failed: {fp.name} | {e}")

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

# =========================
# LOAD
# =========================
gm_df = load_all_google_maps(ROOT_DIR)
print("Rows:", gm_df.shape[0])
print("Cols:", gm_df.shape[1])
gm_df.head(3)


Rows: 1016595
Cols: 85


,categoryName,city,location/lat,location/lng,neighborhood,publishedAtDate,stars,street,text,title,...,CategoryName,Location/lat,Location/lng,Neighborhood,Date,Stars,City,Place Name,Street,Region
0,فندق منتجع,بني سار,20.082359,41.450166,NaN,2025-10-13T14:05:31.133Z,5.0,طريق الملك عبدالعزيز,اشكرهم على الاستقبال وحق الاستقبال المصري والل...,اكواخ سار الريفية,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,فندق منتجع,بني سار,20.082359,41.450166,NaN,2025-10-04T12:18:17.557Z,5.0,طريق الملك عبدالعزيز,NaN,اكواخ سار الريفية,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,فندق منتجع,بني سار,20.082359,41.450166,NaN,2025-10-04T11:21:58.022Z,5.0,طريق الملك عبدالعزيز,NaN,اكواخ سار الريفية,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
gm_df[["macro_region","subfolder","place_name"]].drop_duplicates().head(500)

,macro_region,subfolder,place_name
0,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,أكواخ سار الريفية
440,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,القرية الأثرية بالأطاولة
948,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,جبل شدا الأعلى
1196,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الأمير سلطان بن سلمان
2522,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الأمير محمد بن سعود
...,...,...,...
1002130,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...
1008174,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه فلايح عنيزة_final
1009409,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزهات الجال بالشماسية_final
1009960,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزهات الحاجب_final


**These outputs provide an overview of the structure and distribution of the Google Maps data. The first result shows how reviews are distributed across the five main regions of Saudi Arabia, helping us understand the data volume in each region and detect any imbalance before starting text analysis. The second result confirms the hierarchical structure of the dataset (region → city → tourist place), ensuring that the files were loaded correctly and highlighting the diversity of locations included in the data.**

### **Data Quality + Basic Overview**

In [15]:
print("Shape:", gm_df.shape)
gm_df.info()
gm_df.describe(include="all").T.head(10)

Shape: (1016595, 85)
<class 'pandas.DataFrame'>
RangeIndex: 1016595 entries, 0 to 1016594
Data columns (total 85 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   categoryName             531560 non-null   str    
 1   city                     448174 non-null   object 
 2   location/lat             531560 non-null   float64
 3   location/lng             531560 non-null   float64
 4   neighborhood             329363 non-null   object 
 5   publishedAtDate          531560 non-null   str    
 6   stars                    531522 non-null   float64
 7   street                   464826 non-null   object 
 8   text                     381232 non-null   object 
 9   title                    531560 non-null   str    
 10  Text_Orig                542447 non-null   object 
 11  Text                     542447 non-null   object 
 12  Emoji_List               1016595 non-null  str    
 13  Emoji_Count              1016595

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
categoryName,531560,61,متنزه,147997,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,448174,54,الخبر,66895,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location/lat,531560.0,NaN,NaN,NaN,22.305551,3.940002,16.414797,18.241283,20.076306,26.302907,28.446341
location/lng,531560.0,NaN,NaN,NaN,45.843705,3.836455,41.239697,42.328826,44.210328,49.998547,50.224079
neighborhood,329363,84,الفناتير,26183,NaN,NaN,NaN,NaN,NaN,NaN,NaN
publishedAtDate,531560,531466,2017-08-03T09:06:05.010Z,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stars,531522.0,NaN,NaN,NaN,4.236575,1.180683,1.0,4.0,5.0,5.0,5.0
street,464826,169,4657 الحكم الزروقي، 6905,37938,NaN,NaN,NaN,NaN,NaN,NaN,NaN
text,381232,299313,ممتاز,5778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,531560,214,منتزه غابة رغدان,37938,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
before = gm_df.shape[0]
gm_df = gm_df.drop_duplicates()
after = gm_df.shape[0]

print("Removed duplicates:", before - after)


Removed duplicates: 3926


### **Rating Distribution**

In [18]:
import pandas as pd
import plotly.express as px

# =========================================================
# 1) توحيد عمود النجوم
# =========================================================
star_cols = [c for c in gm_df.columns if c.strip().lower() == "stars"]

if len(star_cols) == 0:
    raise ValueError("No Stars column found in gm_df.")

elif len(star_cols) == 1:
    gm_df["Stars_unified"] = gm_df[star_cols[0]]

else:
    # إذا فيه أكثر من عمود، خذ أول قيمة غير فارغة
    gm_df["Stars_unified"] = gm_df[star_cols].bfill(axis=1).iloc[:, 0]

# تحويل لأرقام
gm_df["Stars_unified"] = pd.to_numeric(gm_df["Stars_unified"], errors="coerce")

# الاحتفاظ فقط بالقيم الصحيحة من 1 إلى 5
stars_clean = gm_df.loc[gm_df["Stars_unified"].between(1, 5), "Stars_unified"]

# =========================================================
# 2) تجهيز بيانات الرسم
# =========================================================
stars_counts = (
    stars_clean.value_counts()
    .sort_index()
    .rename_axis("Stars")
    .reset_index(name="Count")
)

stars_counts["Stars"] = stars_counts["Stars"].astype(int).astype(str)
stars_counts["Percentage"] = (stars_counts["Count"] / stars_counts["Count"].sum() * 100).round(2)
stars_counts["Label"] = stars_counts["Count"].astype(str) + " reviews"

# =========================================================
# 3) الرسم التفاعلي
# =========================================================
fig = px.bar(
    stars_counts,
    x="Stars",
    y="Count",
    text="Count",
    hover_data={"Percentage": True, "Count": True, "Stars": True},
    title="Interactive Distribution of Ratings",
)

fig.update_traces(
    textposition="outside",
    hovertemplate=
    "<b>Stars:</b> %{x}<br>" +
    "<b>Count:</b> %{y}<br>" +
    "<b>Percentage:</b> %{customdata[0]}%<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Stars",
    yaxis_title="Number of Reviews",
    title_x=0.5,
    font=dict(size=14),
    bargap=0.25,
    hoverlabel=dict(font_size=13),
    height=500
)

fig.show()

### **Basic Text Metrics**

In [19]:
TEXT_COL = "Text_TR"

gm_df["char_length"] = gm_df[TEXT_COL].astype(str).str.len()
gm_df["word_count"] = gm_df[TEXT_COL].astype(str).str.split().str.len()

In [20]:
! pip install seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1) تجهيز البيانات
# =========================================================
if "word_count" not in gm_df.columns:
    raise ValueError("Column 'word_count' not found in gm_df.")

gm_df["word_count"] = pd.to_numeric(gm_df["word_count"], errors="coerce")
wc = gm_df["word_count"].dropna()

# قص القيم المتطرفة
upper_limit = wc.quantile(0.99)
trimmed = wc[wc <= upper_limit]

# =========================================================
# 2) إعداد bins
# =========================================================
bin_width = 5
bins = np.arange(0, trimmed.max() + bin_width, bin_width)

hist_counts, bin_edges = np.histogram(trimmed, bins=bins)

hist_df = pd.DataFrame({
    "Bin_Start": bin_edges[:-1],
    "Bin_End": bin_edges[1:],
    "Count": hist_counts
})

hist_df["Range"] = hist_df.apply(
    lambda r: f"{int(r['Bin_Start'])}–{int(r['Bin_End'])-1}", axis=1
)

# =========================================================
# 3) حساب المتوسط والوسيط
# =========================================================
mean_wc = trimmed.mean()
median_wc = trimmed.median()

# =========================================================
# 4) الرسم التفاعلي
# =========================================================
fig = go.Figure()

fig.add_trace(go.Bar(
    x=hist_df["Range"],
    y=hist_df["Count"],
    text=hist_df["Count"],
    textposition="outside",
    hovertemplate=
    "<b>Word range:</b> %{x}<br>" +
    "<b>Number of reviews:</b> %{y}<br>" +
    "<extra></extra>",
    marker=dict(
        line=dict(width=1.2, color="rgba(0,0,0,0.35)")
    )
))

# خطوط Mean و Median
fig.add_vline(
    x=mean_wc / bin_width,
    line_width=3,
    line_dash="dash",
    line_color="black"
)

fig.add_vline(
    x=median_wc / bin_width,
    line_width=3,
    line_dash="dot",
    line_color="black"
)

# النص فوق يمين الشارت
fig.add_annotation(
    x=1,
    y=1,
    xref="paper",
    yref="paper",
    text=f"<b>Mean = {mean_wc:.1f}<br>Median = {median_wc:.1f}</b>",
    showarrow=False,
    align="right",
    xanchor="right",
    yanchor="top",
    font=dict(size=14),
    bgcolor="rgba(255,255,255,0.7)"
)

# =========================================================
# 5) تنسيق الشكل
# =========================================================
fig.update_layout(
    title="Text Review Length",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Number of Words (Grouped in bins of 5)",
    yaxis_title="Number of Reviews",
    bargap=0.08,
    height=600,
    font=dict(size=14)
)

fig.update_xaxes(tickangle=-45)
fig.update_yaxes(showgrid=True)

fig.show()

**This chart shows the distribution of review lengths (in words) . Each bar represents a range of 5 words, making the pattern easier to interpret. The majority of reviews are short, with most comments containing fewer than 20 words, while longer reviews appear less frequently. The dashed lines indicate the mean and median review length, highlighting the right-skewed nature of the data.**

### **Top Words Analysis**

In [10]:
! pip install arabic-reshaper python-bidi

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from collections import Counter

TEXT_COL = "Text_Base"

# ناخذ العمود فقط ونشيل القيم الفارغة
text_series = gm_df[TEXT_COL].dropna()

# نفك الكلمات
all_words = (
    text_series
    .astype(str)
    .str.split()
    .explode()
)

# نحذف أي بقايا "nan" نصية
all_words = all_words[all_words.str.lower() != "nan"]

# نحسب الأكثر تكرارًا
top_words = Counter(all_words).most_common(20)

top_words_df = pd.DataFrame(top_words, columns=["word", "frequency"])
top_words_df


,word,frequency
0,من,127143
1,جدا,115281
2,جميل,110936
3,في,101920
4,مكان,89377
5,المكان,66450
6,و,59719
7,علي,53972
8,فيه,43988
9,رائع,39178


###  **Next: Top Words after removing Stopwords**

In [13]:
AR_STOPWORDS = set("""
و في من على الى إلى عن مع ما لا نعم بس لكن او أو اذا إذا ان إن انه إنها هذا هذه ذلك تلك ثم جدا جدًا مرة مرا
كل كان تكون يكون كانت كانوا يكونون عند عنده عندها عندهم فيه فيها فيهاً فيه
""".split())

In [28]:
import pandas as pd
from collections import Counter
import plotly.graph_objects as go

# =========================================================
# SETTINGS
# =========================================================
TEXT_COL = "Text_Base"   # غيّرها إذا عندك اسم مختلف
TOP_N = 20

# =========================================================
# 1) توحيد عمود النجوم
# =========================================================
star_cols = [c for c in gm_df.columns if c.strip().lower() == "stars"]

if len(star_cols) == 0:
    raise ValueError("No Stars column found in gm_df.")
elif len(star_cols) == 1:
    gm_df["Stars_unified"] = gm_df[star_cols[0]]
else:
    gm_df["Stars_unified"] = gm_df[star_cols].bfill(axis=1).iloc[:, 0]

gm_df["Stars_unified"] = pd.to_numeric(gm_df["Stars_unified"], errors="coerce")

# =========================================================
# 2) دوال مساعدة
# =========================================================
def keep_arabic_only(token: str) -> str:
    # يبقي الحروف العربية فقط ويستبعد الإنجليزية والأرقام والرموز
    return "".join(ch for ch in str(token) if '\u0600' <= ch <= '\u06FF')

# =========================================================
# 3) كلمات مستبعدة
# =========================================================
AR_STOPWORDS = set("""
و في من على الى إلى عن مع ما لا نعم بس لكن او أو اذا إذا ان إن انه إنها هذا هذه ذلك تلك ثم
جدا جدًا مرة مرا كل كان كانت يكون تكون كانوا يكونون عند عنده عندها عندهم فيه فيها هنا هناك
كما ايضا أيضًا فقط قد بعد قبل بين حتى ضمن حول تحت فوق داخل خارج الذي التي الذين اللواتي
له لها لهم هن يا أي اي أم بل حيث حيثما فقط جدا جداً مرة مره شيء شي
الخ ال بعد قبل اكثر أقل أقلها أكثرها الى الى من عن على في ثم او أم
""".split())

RELIGIOUS_EXCLUDE = set("""
الله لله بالله والله تالله اللهم إله الاله الإله ربي رب الرب ياالله يااللهم
""".split())

GENERIC_EXCLUDE = set("""
مكان المكان جدا جداً مره مرة شيء شي اليوم امس بكرا ايضا أيضًا بصراحة الصراحه مررره
اللي الي انه انها أنهم انهن هذي هذيك هذه هذا هناك هنا كله كلها كلهم
مرة مره جدا جداً حلو جميل ممتاز سيء سيئة رائع رائعه روعة
""".split())

ALL_EXCLUDE = AR_STOPWORDS | RELIGIOUS_EXCLUDE | GENERIC_EXCLUDE

# =========================================================
# 4) استخراج التوب كلمات
# =========================================================
def extract_top_words(df_subset: pd.DataFrame, text_col: str, top_n: int = 20) -> pd.DataFrame:
    text_series = df_subset[text_col].dropna().astype(str)

    words = text_series.str.split().explode().dropna()
    words = words.map(keep_arabic_only)
    words = words.str.strip()

    words = words[
        (words.str.len() >= 2) &
        (~words.isin(ALL_EXCLUDE))
    ]

    # استبعاد بقايا غير مفيدة
    words = words[~words.isin([
        "ال", "وال", "بال", "لل", "ثم", "قد", "او", "أي", "اي",
        "هذا", "هذه", "ذلك", "تلك", "التي", "الذي"
    ])]

    top_df = pd.DataFrame(
        Counter(words).most_common(top_n),
        columns=["word", "frequency"]
    )

    return top_df

# =========================================================
# 5) رسم الشارت التفاعلي
# =========================================================
def plot_top_words_interactive(top_df: pd.DataFrame, title: str):
    if top_df.empty:
        print(f"No data available for: {title}")
        return

    plot_df = top_df.sort_values("frequency", ascending=True).copy()

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=plot_df["frequency"],
        y=plot_df["word"],   # بدون fix_arabic عشان ما تنعكس
        orientation="h",
        text=plot_df["frequency"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        hovertemplate=
        "<b>Word:</b> %{y}<br>" +
        "<b>Frequency:</b> %{x:,}<extra></extra>",
        marker=dict(
            line=dict(width=1.2, color="rgba(0,0,0,0.35)")
        )
    ))

    fig.update_layout(
        title=title,
        title_x=0.5,
        template="plotly_white",
        xaxis_title="Frequency",
        yaxis_title="Word",
        height=700,
        font=dict(size=14),
        margin=dict(l=140, r=40, t=80, b=60)
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=False, autorange="reversed")

    fig.show()

# =========================================================
# 6) تقسيم التعليقات حسب النجوم
# =========================================================
positive_df = gm_df[gm_df["Stars_unified"].isin([4, 5])].copy()
negative_df = gm_df[gm_df["Stars_unified"].isin([1, 2])].copy()

print("Positive reviews:", positive_df.shape[0])
print("Negative reviews:", negative_df.shape[0])

# =========================================================
# 7) استخراج أعلى الكلمات
# =========================================================
top_positive_words = extract_top_words(positive_df, TEXT_COL, TOP_N)
top_negative_words = extract_top_words(negative_df, TEXT_COL, TOP_N)

print("\nTop Positive Words:")
print(top_positive_words)

print("\nTop Negative Words:")
print(top_negative_words)

# =========================================================
# 8) عرض الشارتين
# =========================================================
plot_top_words_interactive(
    top_positive_words,
    "Top 20 Most Frequent Arabic Words in Positive Reviews"
)

plot_top_words_interactive(
    top_negative_words,
    "Top 20 Most Frequent Arabic Words in Negative Reviews"
)

Positive reviews: 809200
Negative reviews: 100397

Top Positive Words:
       word  frequency
0       علي      38738
1     جميلة      18519
2     جميله      16361
3       جيد      14324
4      افضل      14072
5      يوجد      13706
6   للاطفال      13461
7      انصح      11655
8   الزيارة      11630
9     يستحق      10942
10     اجمل      10882
11    رائعة      10289
12     ولكن      10163
13    منتزه       9867
14   الدخول       9858
15    حديقة       9348
16    العاب       9139
17     روعه       9135
18     ريال       9126
19    تجربة       8385

Top Negative Words:
       word  frequency
0       علي       9738
1       ولا       8294
2       غير       6991
3      ريال       5037
4    الدخول       4906
5      يوجد       4171
6     للاسف       4052
7       حتي       2985
8     يحتاج       2905
9      ولكن       2796
10    مبالغ       2748
11       مو       2499
12       لم       2380
13      الا       2336
14  الاطفال       2212
15  الالعاب       2205
16     عشان       2203
17     انصح

**This graph illustrates the most frequently occurring words in the Text_Base column after cleaning the text and removing common stopwords. The distribution reflects the words most closely associated with the review content, helping to understand the recurring themes and impressions among visitors. This analysis contributes to uncovering the key concepts that characterize the tourist experience within the studied data.**

### **Top Bigrams**

In [29]:
import re
import pandas as pd
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer

# =========================================================
# SETTINGS
# =========================================================
TEXT_COL = "Text_Base"
TOP_N = 20

# =========================================================
# 1) تحديد عمود المنطقة
# =========================================================
if "macro_region" in gm_df.columns:
    REGION_COL = "macro_region"
elif "region" in gm_df.columns:
    REGION_COL = "region"
else:
    raise ValueError("No region column found. Expected 'macro_region' or 'region'.")

# =========================================================
# 2) تنظيف اسم المنطقة
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def clean_region_name(region_name: str) -> str:
    region_name = str(region_name).strip()
    m = PAREN_RE.search(region_name)
    if m:
        return m.group(1).strip()
    return region_name

gm_df["region_clean"] = gm_df[REGION_COL].astype(str).apply(clean_region_name)

# =========================================================
# 3) قوائم الاستبعاد
# =========================================================
AR_STOPWORDS = set("""
و في من على الى إلى عن مع ما لا نعم بس لكن او أو اذا إذا ان إن انه إنها هذا هذه ذلك تلك ثم
جدا جدًا مرة مرا كل كان كانت يكون تكون كانوا يكونون عند عنده عندها عندهم فيه فيها هنا هناك
كما ايضا أيضًا فقط قد بعد قبل بين حتى ضمن حول تحت فوق داخل خارج الذي التي الذين اللواتي
له لها لهم هن يا أي اي أم بل حيث حيثما فقط جدا جداً مرة مره شيء شي
الخ ال بعد قبل اكثر أقل أكثرها أقلها
""".split())

RELIGIOUS_EXCLUDE = set("""
الله لله بالله والله تالله اللهم إله الاله الإله ربي رب الرب ياالله يااللهم
""".split())

GENERIC_EXCLUDE = set("""
مكان المكان جدا جداً مره مرة شيء شي اليوم امس بكرا ايضا أيضًا بصراحة الصراحه مررره
اللي الي انه انها أنهم انهن هذي هذيك هذه هذا هناك هنا كله كلها كلهم
جميل جميلة ممتاز ممتازة رائع رائعة روعة سيء سيئة حلو حلوة
""".split())

ALL_EXCLUDE = AR_STOPWORDS | RELIGIOUS_EXCLUDE | GENERIC_EXCLUDE

# =========================================================
# 4) دوال تنظيف النص
# =========================================================
def keep_arabic_only(token: str) -> str:
    return "".join(ch for ch in str(token) if '\u0600' <= ch <= '\u06FF')

def clean_text_for_bigrams(text: str) -> str:
    tokens = str(text).split()
    cleaned_tokens = []

    for tok in tokens:
        tok = keep_arabic_only(tok).strip()

        if len(tok) < 2:
            continue
        if tok in ALL_EXCLUDE:
            continue
        if tok in {"ال", "وال", "بال", "لل", "ثم", "قد", "او", "أي", "اي", "هذا", "هذه", "ذلك", "التي", "الذي"}:
            continue

        cleaned_tokens.append(tok)

    return " ".join(cleaned_tokens)

# =========================================================
# 5) استخراج Top Bigrams لمنطقة واحدة
# =========================================================
def extract_top_bigrams_for_region(df_region: pd.DataFrame, text_col: str, top_n: int = 20) -> pd.DataFrame:
    texts = df_region[text_col].dropna().astype(str)

    # تنظيف النصوص
    cleaned_texts = texts.apply(clean_text_for_bigrams)
    cleaned_texts = cleaned_texts[cleaned_texts.str.strip() != ""]

    if cleaned_texts.empty:
        return pd.DataFrame(columns=["bigram", "frequency"])

    vectorizer = CountVectorizer(
        ngram_range=(2, 2),
        max_features=top_n,
        token_pattern=r"(?u)\b\w+\b"
    )

    X = vectorizer.fit_transform(cleaned_texts)

    bigrams = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1

    bigrams_df = pd.DataFrame({
        "bigram": bigrams,
        "frequency": counts
    }).sort_values("frequency", ascending=False)

    return bigrams_df

# =========================================================
# 6) رسم شارت تفاعلي
# =========================================================
def plot_region_bigrams_interactive(bigrams_df: pd.DataFrame, region_name: str):
    if bigrams_df.empty:
        print(f"No bigrams available for region: {region_name}")
        return

    plot_df = bigrams_df.sort_values("frequency", ascending=True).copy()

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=plot_df["frequency"],
        y=plot_df["bigram"],
        orientation="h",
        text=plot_df["frequency"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        hovertemplate=
        "<b>Bigram:</b> %{y}<br>" +
        "<b>Frequency:</b> %{x:,}<extra></extra>",
        marker=dict(
            line=dict(width=1.2, color="rgba(0,0,0,0.35)")
        )
    ))

    fig.update_layout(
        title=f"Top {TOP_N} Most Frequent Bigrams - {region_name}",
        title_x=0.5,
        template="plotly_white",
        xaxis_title="Frequency",
        yaxis_title="Bigram",
        height=750,
        font=dict(size=14),
        margin=dict(l=180, r=40, t=80, b=60)
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=False)

    fig.show()

# =========================================================
# 7) تنفيذ الشارت لكل منطقة
# =========================================================
regions = sorted(gm_df["region_clean"].dropna().unique())

print("Regions found:", regions)

for region in regions:
    region_df = gm_df[gm_df["region_clean"] == region].copy()

    bigrams_df = extract_top_bigrams_for_region(
        region_df,
        text_col=TEXT_COL,
        top_n=TOP_N
    )

    print(f"\nRegion: {region}")
    print(bigrams_df.head(10))

    plot_region_bigrams_interactive(bigrams_df, region)

Regions found: ['المنطقة الجنوبية', 'المنطقة الشرقية', 'المنطقة الشمالية', 'المنطقة الغربية', 'المنطقة الوسطى']

Region: المنطقة الجنوبية
           bigram  frequency
16  يستحق الزيارة       2216
6    دورات المياه       2022
7      دورات مياه       1575
17  يستحق الزياره       1200
3    انصح بزيارته       1126
1     العاب اطفال        966
13       ولا يوجد        901
0    اجمل الاماكن        881
8     رسوم الدخول        831
5   تستحق الزيارة        823



Region: المنطقة الشرقية
           bigram  frequency
17  يستحق الزيارة       1472
10      علي البحر       1006
4    انصح بزيارته        791
6    دورات المياه        782
18  يستحق الزياره        754
16      يحتوي علي        683
5        بشكل عام        680
9        سوق شعبي        635
7      دورات مياه        614
19        يوجد به        594



Region: المنطقة الشمالية
              bigram  frequency
18     يستحق الزيارة        540
19     يستحق الزياره        307
8           بشكل عام        285
5       انصح بزيارته        264
6            انصح به        227
2   العربية السعودية        211
3    المملكة العربية        187
12       علي الاطلاق        178
7         بانيان تري        178
10     تستحق الزيارة        176



Region: المنطقة الغربية
             bigram  frequency
13        عليه وسلم        857
10         صلي عليه        818
17    يستحق الزيارة        643
7      دورات المياه        585
8        دورات مياه        531
3   المدينة المنورة        531
4      انصح بزيارته        378
12        علي البحر        342
1       العاب اطفال        319
18    يستحق الزياره        318



Region: المنطقة الوسطى
              bigram  frequency
18     يستحق الزيارة       1335
9       دورات المياه        975
5   العربية السعودية        774
8           بشكل عام        761
10        دورات مياه        725
6    المملكة العربية        674
19     يستحق الزياره        660
7       انصح بزيارته        624
11          سوق شعبي        601
2        العاب اطفال        592


**This graph illustrates the most frequently occurring binary phrases (bigrams) in the Text_Base column, where words were analyzed as two consecutive words to extract the most common context in reviews. This analysis helps uncover recurring linguistic patterns and prominent themes that visitors focus on, providing a deeper understanding of comment content compared to analyzing single words alone.**

In [31]:
import re
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import CountVectorizer

# =========================================================
# SETTINGS
# =========================================================
TEXT_COL = "Text_Base"
TOP_N = 20

# =========================================================
# 1) توحيد عمود النجوم
# =========================================================
star_cols = [c for c in gm_df.columns if c.strip().lower() == "stars"]

if len(star_cols) == 0:
    raise ValueError("No Stars column found.")
elif len(star_cols) == 1:
    gm_df["Stars_unified"] = gm_df[star_cols[0]]
else:
    gm_df["Stars_unified"] = gm_df[star_cols].bfill(axis=1).iloc[:, 0]

gm_df["Stars_unified"] = pd.to_numeric(gm_df["Stars_unified"], errors="coerce")

# =========================================================
# 2) تحديد عمود المنطقة
# =========================================================
if "macro_region" in gm_df.columns:
    REGION_COL = "macro_region"
elif "region" in gm_df.columns:
    REGION_COL = "region"
else:
    raise ValueError("No region column found.")

# تنظيف اسم المنطقة
def clean_region_name(x):
    x = str(x)
    m = re.search(r"\((.*?)\)", x)
    return m.group(1) if m else x

gm_df["region_clean"] = gm_df[REGION_COL].apply(clean_region_name)

# =========================================================
# 3) كلمات مستبعدة
# =========================================================
AR_STOPWORDS = set("""
و في من على الى إلى عن مع ما لا نعم بس لكن او أو اذا إذا ان إن انه إنها هذا هذه ذلك تلك ثم
جدا مرة كل كان كانت يكون تكون كانوا عند فيه هنا هناك كما ايضا فقط قد بعد قبل بين حتى
""".split())

RELIGIOUS = set("""
الله لله بالله اللهم إله الاله ربي رب
""".split())

GENERIC = set("""
مكان المكان شيء شي مرة مره جدا جداً اليوم امس بكرا بصراحة
جميل جميلة ممتاز رائع روعة سيء حلو
زيارة الزيارة يستحق تستحق انصح يوجد يحتوي
""".split())

ALL_EXCLUDE = AR_STOPWORDS | RELIGIOUS | GENERIC

# =========================================================
# 4) تنظيف النص
# =========================================================
def clean_text(text):
    tokens = str(text).split()
    cleaned = []

    for tok in tokens:
        tok = "".join(ch for ch in tok if '\u0600' <= ch <= '\u06FF')

        if len(tok) < 2:
            continue
        if tok in ALL_EXCLUDE:
            continue

        cleaned.append(tok)

    return " ".join(cleaned)

# =========================================================
# 5) استخراج Bigrams
# =========================================================
def get_bigrams(df):
    texts = df[TEXT_COL].dropna().astype(str)
    texts = texts.apply(clean_text)
    texts = texts[texts != ""]

    if texts.empty:
        return pd.DataFrame(columns=["bigram", "frequency"])

    vec = CountVectorizer(ngram_range=(2,2), max_features=TOP_N)
    X = vec.fit_transform(texts)

    bigrams = vec.get_feature_names_out()
    counts = X.sum(axis=0).A1

    return pd.DataFrame({
        "bigram": bigrams,
        "frequency": counts
    }).sort_values("frequency", ascending=False)

# =========================================================
# 6) الرسم
# =========================================================
def plot_region(region_df, region_name):

    pos = region_df[region_df["Stars_unified"].isin([4,5])]
    neg = region_df[region_df["Stars_unified"].isin([1,2])]

    pos_bi = get_bigrams(pos)
    neg_bi = get_bigrams(neg)

    pos_bi = pos_bi.sort_values("frequency", ascending=True)
    neg_bi = neg_bi.sort_values("frequency", ascending=True)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Negative (1–2)", "Positive (4–5)")
    )

    # 🔴 سلبي
    fig.add_trace(
        go.Bar(
            x=neg_bi["frequency"],
            y=neg_bi["bigram"],
            orientation="h",
            text=neg_bi["frequency"],
            textposition="outside",
            marker=dict(color="crimson")
        ),
        row=1, col=1
    )

    # 🔵 إيجابي
    fig.add_trace(
        go.Bar(
            x=pos_bi["frequency"],
            y=pos_bi["bigram"],
            orientation="h",
            text=pos_bi["frequency"],
            textposition="outside",
            marker=dict(color="royalblue")
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=f"Bigrams by Sentiment - {region_name}",
        height=800,
        width=1500,
        showlegend=False
    )

    fig.show()

# =========================================================
# 7) التشغيل
# =========================================================
regions = gm_df["region_clean"].dropna().unique()

for r in regions:
    df_r = gm_df[gm_df["region_clean"] == r]
    print("Processing:", r)
    plot_region(df_r, r)

Processing:  المنطقة الجنوبية 


Processing:  المنطقة الشرقية 


Processing:  المنطقة الشمالية 


Processing:  المنطقة الغربية 


Processing:  المنطقة الوسطى 


In [32]:
import re
import pandas as pd
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer

# =========================================================
# SETTINGS
# =========================================================
TEXT_COL = "Text_Base"
TOP_N = 20

# =========================================================
# 1) توحيد عمود النجوم
# =========================================================
star_cols = [c for c in gm_df.columns if c.strip().lower() == "stars"]

if len(star_cols) == 0:
    raise ValueError("No Stars column found.")
elif len(star_cols) == 1:
    gm_df["Stars_unified"] = gm_df[star_cols[0]]
else:
    gm_df["Stars_unified"] = gm_df[star_cols].bfill(axis=1).iloc[:, 0]

gm_df["Stars_unified"] = pd.to_numeric(gm_df["Stars_unified"], errors="coerce")

# =========================================================
# 2) تحديد عمود المنطقة
# =========================================================
if "macro_region" in gm_df.columns:
    REGION_COL = "macro_region"
elif "region" in gm_df.columns:
    REGION_COL = "region"
else:
    raise ValueError("No region column found.")

# تنظيف اسم المنطقة
def clean_region_name(x):
    x = str(x)
    m = re.search(r"\((.*?)\)", x)
    return m.group(1) if m else x

gm_df["region_clean"] = gm_df[REGION_COL].apply(clean_region_name)

# =========================================================
# 3) كلمات مستبعدة
# =========================================================
AR_STOPWORDS = set("""
و في من على الى إلى عن مع ما لا نعم بس لكن او أو اذا إذا ان إن انه إنها هذا هذه ذلك تلك ثم
جدا مرة كل كان كانت يكون تكون كانوا عند فيه هنا هناك كما ايضا فقط قد بعد قبل بين حتى
""".split())

RELIGIOUS = set("""
الله لله بالله اللهم إله الاله ربي رب
""".split())

GENERIC = set("""
مكان المكان شيء شي مرة مره جدا جداً اليوم امس بكرا بصراحة
جميل جميلة ممتاز رائع روعة سيء حلو
زيارة الزيارة يستحق تستحق انصح يوجد يحتوي
""".split())

ALL_EXCLUDE = AR_STOPWORDS | RELIGIOUS | GENERIC

# =========================================================
# 4) تنظيف النص
# =========================================================
def clean_text(text):
    tokens = str(text).split()
    cleaned = []

    for tok in tokens:
        tok = "".join(ch for ch in tok if '\u0600' <= ch <= '\u06FF')

        if len(tok) < 2:
            continue
        if tok in ALL_EXCLUDE:
            continue

        cleaned.append(tok)

    return " ".join(cleaned)

# =========================================================
# 5) استخراج Bigrams
# =========================================================
def get_bigrams(df):
    texts = df[TEXT_COL].dropna().astype(str)
    texts = texts.apply(clean_text)
    texts = texts[texts != ""]

    if texts.empty:
        return pd.DataFrame(columns=["bigram", "frequency"])

    vec = CountVectorizer(ngram_range=(2,2), max_features=TOP_N)
    X = vec.fit_transform(texts)

    bigrams = vec.get_feature_names_out()
    counts = X.sum(axis=0).A1

    return pd.DataFrame({
        "bigram": bigrams,
        "frequency": counts
    }).sort_values("frequency", ascending=False)

# =========================================================
# 6) الرسم (Neutral فقط)
# =========================================================
def plot_neutral(region_df, region_name):

    neutral = region_df[region_df["Stars_unified"] == 3]

    bigrams = get_bigrams(neutral)
    bigrams = bigrams.sort_values("frequency", ascending=True)

    if bigrams.empty:
        print(f"No neutral data in {region_name}")
        return

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=bigrams["frequency"],
            y=bigrams["bigram"],
            orientation="h",
            text=bigrams["frequency"],
            textposition="outside",
            marker=dict(
                color="gray"  # ⚪ محايد
            )
        )
    )

    fig.update_layout(
        title=f"Top {TOP_N} Neutral Bigrams (Stars = 3) - {region_name}",
        height=800,
        width=900,
        template="plotly_white"
    )

    fig.show()

# =========================================================
# 7) التشغيل
# =========================================================
regions = gm_df["region_clean"].dropna().unique()

for r in regions:
    df_r = gm_df[gm_df["region_clean"] == r]
    print("Processing:", r)
    plot_neutral(df_r, r)

Processing:  المنطقة الجنوبية 


Processing:  المنطقة الشرقية 


Processing:  المنطقة الشمالية 


Processing:  المنطقة الغربية 


Processing:  المنطقة الوسطى 


### **Top Bigrams by Stars (Positive vs Negative Comparison)**

In [33]:
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1) توحيد عمود النجوم (حل مشكلة كابتل/سمول)
# =========================================================
star_cols = [c for c in gm_df.columns if c.strip().lower() == "stars"]

if len(star_cols) == 0:
    raise ValueError("No Stars column found.")
elif len(star_cols) == 1:
    gm_df["Stars_unified"] = gm_df[star_cols[0]]
else:
    gm_df["Stars_unified"] = gm_df[star_cols].bfill(axis=1).iloc[:, 0]

gm_df["Stars_unified"] = pd.to_numeric(gm_df["Stars_unified"], errors="coerce")

# =========================================================
# 2) تقسيم البيانات
# =========================================================
positive_df = gm_df[gm_df["Stars_unified"] >= 4]
negative_df = gm_df[gm_df["Stars_unified"] <= 2]
neutral_df  = gm_df[gm_df["Stars_unified"] == 3]

# =========================================================
# 3) حساب الأعداد والنسب
# =========================================================
counts = [
    len(positive_df),
    len(neutral_df),
    len(negative_df)
]

labels = ["Positive", "Neutral", "Negative"]

total = sum(counts)
percentages = [(c / total) * 100 if total > 0 else 0 for c in counts]

# =========================================================
# 4) رسم Pie Chart احترافي
# =========================================================
fig = go.Figure(
    data=[
        go.Pie(
            labels=labels,
            values=counts,
            hole=0.45,  # Donut style 🔥
            textinfo="label+percent",
            textfont=dict(size=16),
            marker=dict(
                colors=[
                    "royalblue",  # إيجابي 🔵
                    "lightgray",  # محايد ⚪
                    "crimson"     # سلبي 🔴
                ],
                line=dict(color="white", width=2)
            ),
            hovertemplate=
                "<b>%{label}</b><br>" +
                "Count: %{value:,}<br>" +
                "Percentage: %{percent}<extra></extra>"
        )
    ]
)

# =========================================================
# 5) تحسين الشكل
# =========================================================
fig.update_layout(
    title="Sentiment Distribution Based on Stars",
    title_x=0.5,
    template="plotly_white",
    height=600,
    width=800,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    )
)

fig.show()

# =========================================================
# 6) طباعة النسب (اختياري)
# =========================================================
for lbl, cnt, pct in zip(labels, counts, percentages):
    print(f"{lbl}: {cnt:,} ({pct:.2f}%)")

Positive: 809,200 (79.94%)
Neutral: 102,692 (10.14%)
Negative: 100,397 (9.92%)


In [36]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_bigrams(df, top_n=15):
    texts = df[TEXT_COL].dropna().astype(str)

    vectorizer = CountVectorizer(
        ngram_range=(2,2),
        max_features=top_n
    )

    X = vectorizer.fit_transform(texts)

    bigrams = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1

    result_df = pd.DataFrame({
        "bigram": bigrams,
        "frequency": counts
    }).sort_values(by="frequency", ascending=False)

    return result_df

In [37]:
pos_bigrams = get_top_bigrams(positive_df, top_n=15)
neg_bigrams = get_top_bigrams(negative_df, top_n=15)

pos_bigrams.head()
neg_bigrams.head()

,bigram,frequency
9,لا يوجد,1967
3,جدا جدا,1911
2,المكان جميل,1449
11,مبالغ فيها,1423
13,ولا يوجد,1128


In [39]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1) تجهيز البيانات
# =========================================================
pos_plot = pos_bigrams.sort_values("frequency", ascending=True).copy()
neg_plot = neg_bigrams.sort_values("frequency", ascending=True).copy()

# لا نستخدم fix_arabic مع Plotly حتى لا تنعكس العربية
if "bigram_fixed" in pos_plot.columns:
    pos_plot = pos_plot.drop(columns=["bigram_fixed"])

if "bigram_fixed" in neg_plot.columns:
    neg_plot = neg_plot.drop(columns=["bigram_fixed"])

# =========================================================
# 2) الرسم التفاعلي
# =========================================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Negative Reviews (1–2 Stars)",
        "Positive Reviews (4–5 Stars)"
    ),
    horizontal_spacing=0.12
)

# 🔴 Negative - Left
fig.add_trace(
    go.Bar(
        x=neg_plot["frequency"],
        y=neg_plot["bigram"],
        orientation="h",
        text=neg_plot["frequency"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        marker=dict(
            color="crimson",
            line=dict(width=1.2, color="rgba(0,0,0,0.35)")
        ),
        hovertemplate=
        "<b>Bigram:</b> %{y}<br>" +
        "<b>Frequency:</b> %{x:,}<extra></extra>"
    ),
    row=1, col=1
)

# 🔵 Positive - Right
fig.add_trace(
    go.Bar(
        x=pos_plot["frequency"],
        y=pos_plot["bigram"],
        orientation="h",
        text=pos_plot["frequency"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        marker=dict(
            color="royalblue",
            line=dict(width=1.2, color="rgba(0,0,0,0.35)")
        ),
        hovertemplate=
        "<b>Bigram:</b> %{y}<br>" +
        "<b>Frequency:</b> %{x:,}<extra></extra>"
    ),
    row=1, col=2
)

# =========================================================
# 3) تنسيق الشكل
# =========================================================
fig.update_layout(
    title="Top Bigrams by Sentiment",
    title_x=0.5,
    template="plotly_white",
    height=850,
    width=1600,
    showlegend=False,
    font=dict(size=14),
    margin=dict(l=80, r=80, t=100, b=60)
)

fig.update_xaxes(title_text="Frequency", showgrid=True, row=1, col=1)
fig.update_xaxes(title_text="Frequency", showgrid=True, row=1, col=2)

fig.update_yaxes(title_text="Bigram", showgrid=False, row=1, col=1, autorange="reversed")
fig.update_yaxes(title_text="Bigram", showgrid=False, row=1, col=2, autorange="reversed")

fig.show()

**This chart compares the most frequent bigrams in positive reviews (4–5 stars) and negative reviews (1–2 stars). The left panel shows that positive reviews commonly include expressions of praise related to beauty, quality, and overall experience. In contrast, the right panel highlights phrases associated with complaints or dissatisfaction in negative reviews. This comparison clearly illustrates the linguistic differences between positive and negative visitor experiences and helps identify the aspects influencing user ratings.**

### **Aspect-Based Analysis**

In [41]:
ASPECTS = {
    "المنظر والطبيعة": ["منظر", "اطلالة", "طبيعة", "جميل"],
    "الأسعار": ["سعر", "اسعار", "غالي", "رخيص", "مبالغ"],
    "النظافة": ["نظيف", "نظافة", "وسخ", "قذر"],
    "الخدمات": ["خدمة", "تعامل", "موظف", "استقبال"],
    "دورات المياه": ["دورات", "حمام", "مياه"],
    "المواقف": ["موقف", "مواقف", "سيارات"],
    "الازدحام": ["زحام", "ازدحام", "مزدحم", "هدوء"]
}


In [42]:
TEXT_COL = "Text_Base"

aspect_counts = {}

for aspect, keywords in ASPECTS.items():
    count = gm_df[TEXT_COL].dropna().astype(str).apply(
        lambda x: any(word in x for word in keywords)
    ).sum()
    
    aspect_counts[aspect] = count

aspect_df = pd.DataFrame.from_dict(
    aspect_counts, orient="index", columns=["frequency"]
).sort_values(by="frequency", ascending=False)

aspect_df

,frequency
المنظر والطبيعة,163374
الأسعار,35327
النظافة,32494
الخدمات,25525
دورات المياه,21441
المواقف,15445
الازدحام,9557


In [43]:
import plotly.graph_objects as go

# =========================================================
# 1) تجهيز البيانات
# =========================================================
aspect_plot = aspect_df.copy().reset_index()

# إذا اسم العمود الأول مو aspect نعيد تسميته
aspect_plot.columns = ["aspect", "frequency"]

# ترتيب من الأقل للأعلى عشان barh يطلع مرتب
aspect_plot = aspect_plot.sort_values("frequency", ascending=True)

# =========================================================
# 2) الرسم التفاعلي
# =========================================================
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=aspect_plot["frequency"],
        y=aspect_plot["aspect"],
        orientation="h",
        text=aspect_plot["frequency"].apply(lambda x: f"{x:,}"),
        textposition="outside",
        marker=dict(
            line=dict(width=1.2, color="rgba(0,0,0,0.35)")
        ),
        hovertemplate=
        "<b>Aspect:</b> %{y}<br>" +
        "<b>Frequency:</b> %{x:,}<extra></extra>"
    )
)

# =========================================================
# 3) تنسيق الشكل
# =========================================================
fig.update_layout(
    title="Aspect-Based Analysis",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Frequency",
    yaxis_title="Aspect",
    height=650,
    width=1100,
    font=dict(size=14),
    margin=dict(l=120, r=40, t=80, b=60)
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=False)

fig.show()

### **This chart illustrates the most discussed aspects in the reviews. The results show that view and scenery are mentioned far more frequently than other aspects, indicating that visual experience is the primary factor influencing visitor evaluations. Price, cleanliness, and service are discussed to a lesser extent, while crowding and parking are mentioned relatively less often. This distribution highlights the key factors shaping overall visitor perceptions and experiences.**